In [ ]:
!pip install pyserial requests


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import serial
import requests
import time

# ---------------- Arduino PORT ----------------
# Make sure 'COM4' matches the port shown in your Arduino IDE
arduino = serial.Serial('COM4', 9600)
time.sleep(2)

# ---------------- LM Studio API ----------------
url = "http://127.0.0.1:1234/v1/chat/completions"

SYSTEM_PROMPT = """
You are a robot security AI.

Rules:
- SAFE = object is far away (greater than 100 cm)
- WARNING = object nearby (between 30 cm and 100 cm)
- ALERT = object too close (less than 30 cm)

Reply ONLY with one word:
SAFE
WARNING
ALERT
"""

# ---------------- AI FUNCTION ----------------
def ask_ai(distance):
    data = {
        "model": "local-model",
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Distance detected: {distance} cm"}
        ],
        "temperature": 0.1
    }

    try:
        response = requests.post(url, json=data, timeout=5)

        # Check if request succeeded
        if response.status_code != 200:
            return f"API ERROR {response.status_code}"

        result = response.json()

        # Safe extraction from JSON
        if "choices" in result:
            ai_reply = result["choices"][0]["message"]["content"].strip()

            # Clean output
            ai_reply = ai_reply.upper()

            # Only allow valid statuses
            if "SAFE" in ai_reply:
                return "SAFE"
            elif "WARNING" in ai_reply:
                return "WARNING"
            elif "ALERT" in ai_reply:
                return "ALERT"

        return "UNKNOWN"

    except Exception as e:
        return f"ERROR ({str(e)})"

# ---------------- MAIN LOOP ----------------
print("AI Robot Bridge Running...")

while True:

    if arduino.in_waiting:

        line = arduino.readline().decode(errors='ignore').strip()

        print("Arduino:", line)

        # Example line:
        # Angle: 92 Distance: 40

        if "Distance:" in line:

            try:
                # Extract distance
                distance_str = line.split("Distance:")[1].strip()
                distance = int(distance_str)

                # Get AI decision
                decision = ask_ai(distance)

                # Display status
                if decision == "SAFE":
                    print("AI Assessment: SAFE ✅")

                elif decision == "WARNING":
                    print("AI Assessment: WARNING ⚠️")

                elif decision == "ALERT":
                    print("AI Assessment: ALERT 🚨")

                else:
                    print(f"AI Assessment: {decision}")

                print("-" * 35)

                # Send status back to Arduino
                arduino.write((decision + "\n").encode())

            except Exception as e:
                print(f"Parsing Error: {e}")

AI Robot Bridge Running...
Arduino: Angle: 92 Distance: 116
AI Assessment: ERROR (HTTPConnectionPool(host='127.0.0.1', port=1234): Read timed out. (read timeout=5))
-----------------------------------
Arduino: Angle: 94 Distance: 116
AI Assessment: SAFE ✅
-----------------------------------
Arduino: Angle: 96 Distance: 117
AI Assessment: SAFE ✅
-----------------------------------
Arduino: Angle: 98 Distance: 118
AI Assessment: SAFE ✅
-----------------------------------
Arduino: Angle: 100 Distance: 7
AI Assessment: ALERT 🚨
-----------------------------------
Arduino: Angle: 102 Distance: 277
AI Assessment: SAFE ✅
-----------------------------------
Arduino: Angle: 104 Distance: 198
AI Assessment: SAFE ✅
-----------------------------------
Arduino: Angle: 106 Distance: 40
AI Assessment: WARNING ⚠️
-----------------------------------
Arduino: Angle: 108 Distance: 74
AI Assessment: WARNING ⚠️
-----------------------------------
Arduino: Angle: 110 Distance: 40
AI Assessment: WARNING ⚠️
--